# DSN Technical Interview Assessment

#Instructions
- Make a copy of this Notebook, rename it and store in your drive.
- Complete your coding assessment in your own copy of the Notebook.
- When you're done, share the link to your version of the notebook with DSN, rememeber to set access to "Anyone with the link"
- Goodluck!

## Assessment Task: Building FastAPI Applications for Local Languages AI Tasks

### Overview
This assessment evaluates your ability to build production-ready FastAPI applications that integrate Large Language Models (LLMs) and local language processing tasks including:
- **Automatic Speech Recognition (ASR)** - Speech-to-Text conversion
- **Machine Translation** - Text translation between local languages
- **Text-to-Speech (TTS)** - Speech synthesis with multilingual support

### Assessment Goals
1. Demonstrate proficiency with FastAPI framework and async programming
2. Integrate third-party AI APIs (Groq for LLMs, Speech API for audio processing)
3. Build scalable, production-ready microservices
4. Handle real-world challenges: error handling, rate limiting, validation
5. Write comprehensive tests and documentation

### Key Technologies
- **FastAPI**: Modern web framework for building APIs with Python
- **Groq API**: Fast LLM inference with free tier (https://console.groq.com)
- **Spitch API**: Audio processing service with free tier (https://spitch.app/)
- **Pydantic**: Data validation using Python type hints
- **pytest**: Testing framework

### Expected Deliverables
1. Fully functional FastAPI application with at least 3 endpoints
2. Working integration with both Groq and Spitch APIs
3. Clear API documentation and error handling

### Deploy with Gradio Server

**Your entire implementation should be built within this Jupyter notebook:**

1. **Install Dependencies** - Install FastAPI, Groq, Spitch SDK, Gradio, and other required libraries.
2. **Define Pydantic Models** - Create request/response data models for validation.
3. **Implement FastAPI Endpoints** - Build translation, transcription, and synthesis endpoints.
4. **Integrate APIs** - Connect to Groq for LLM-based translation and Spitch API for audio tasks.
5. **Create Gradio Interface** - Build a web UI with Gradio that calls your FastAPI endpoints.
6. **Launch Gradio Server** - Generate a **sharable public link** for remote testing.

**Why Gradio?**
- Automatic web UI generation from your Python functions.
- Generates shareable links (no manual API testing needed)
- Evaluators can test your implementation without local setup.
- Real-time API testing through an intuitive interface.
- Perfect for demonstrating functionality live.

In [ ]:
!pip install -q fastapi uvicorn groq httpx pydantic gradio python-multipart spitch gTTS
!pip uninstall uvloop -y 2>/dev/null || true  # uvloop conflicts with Colab's event loop

In [ ]:
import os
import io
import base64
import tempfile
import logging
from typing import Optional
from enum import Enum

import httpx
import gradio as gr
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Query
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field, field_validator
from groq import Groq

# logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('local-languages-api')

# api keys
from google.colab import userdata
GROQ_API_KEY = userdata.get('Groq_Api_Keys')
SPITCH_API_KEY = userdata.get('Spitch_Api_Keys')

if not GROQ_API_KEY:
    print('Groq API key not found! Add it to Colab Secrets')
if not SPITCH_API_KEY:
    print('Spitch API key not found! Add it to Colab Secrets.')

if GROQ_API_KEY and SPITCH_API_KEY:
    print('API keys loaded successfully')

In [ ]:
#supported languages
class SupportedLanguage(str, Enum):
    ENGLISH = "English"
    YORUBA = "Yoruba"
    HAUSA = "Hausa"
    IGBO = "Igbo"
    PIDGIN = "Nigerian Pidgin"
    FRENCH = "French"
    SWAHILI = "Swahili"
    AMHARIC = "Amharic"
    ARABIC = "Arabic"

#translation models
class TranslationRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="Text to translate",
        examples=["Hello, how are you?"]
    )
    source_language: SupportedLanguage = Field(
        ...,
        description="Source language",
        examples=["English"]
    )
    target_language: SupportedLanguage = Field(
        ...,
        description="Target language",
        examples=["Yoruba"]
    )

    @field_validator("text")
    @classmethod
    def text_must_not_be_blank(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Text must contain non-whitespace characters")
        return v.strip()

class TranslationResponse(BaseModel):
    source_language: str
    target_language: str
    original_text: str
    translated_text: str
    model_used: str

#ASR/Transcription Models
class TranscriptionResponse(BaseModel):
    transcribed_text: str
    language: Optional[str] = None
    duration_seconds: Optional[float] = None

#TTS/Synthesis Models
class SynthesisRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=2000,
        description="Text to synthesize into speech"
    )
    language: str = Field(
        default="en",
        description="Language code (e.g. 'en', 'yo', 'ha', 'ig')"
    )

class SynthesisResponse(BaseModel):
    audio_base64: str
    content_type: str
    message: str

#Health Check
class HealthResponse(BaseModel):
    status: str
    version: str
    services: dict

print("Pydantic models defined")


In [ ]:
# SERVICE 1: GROQ handles translation

class GroqTranslationService:

    def __init__(self, api_key: str):
        self.client = Groq(api_key=api_key)
        self.model = 'llama-3.3-70b-versatile'  # 70B -- much better for African languages

    def translate(self, text: str, source_lang: str, target_lang: str) -> str:
        system_prompt = (
            f'You are a professional translator fluent in {source_lang} and {target_lang}. '
            f'Translate the following text from {source_lang} to {target_lang}.\n\n'
            f'Rules:\n'
            f'- Output ONLY the translation, nothing else.\n'
            f'- Do NOT transliterate -- use actual {target_lang} words and grammar.\n'
            f'- Do NOT invent words. If unsure of a word, keep it in the original language.\n'
            f'- Preserve tone, meaning, and cultural context.\n'
            f'- Use proper diacritics/tone marks where required '
            f'(e.g. \u1eb9, \u1ecd, \u1e63 for Yoruba; \u0253, \u0257 for Hausa).'
        )

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': text}
                ],
                temperature=0.2,
                max_tokens=2048,
            )
            translated = response.choices[0].message.content.strip()
            # strip quotes the model might wrap around the output
            if translated.startswith('"') and translated.endswith('"'):
                translated = translated[1:-1]
            return translated

        except Exception as e:
            logger.error(f'Groq translation error: {e}')
            raise HTTPException(status_code=502, detail=f'Translation service error: {str(e)}')



# SERVICE 2: SPITCH handles audio (ASR + TTS)

from spitch import Spitch

class SpitchAudioService:

    def __init__(self, api_key: str):
        self.client = Spitch(api_key=api_key)

    def transcribe(self, audio_bytes: bytes, language: str = 'en') -> dict:
        """Send audio to Spitch and get back text."""
        try:
            response = self.client.speech.transcribe(
                language=language,
                content=audio_bytes
            )
            return {
                'transcribed_text': response.text,
                'language': language,
            }
        except Exception as e:
            logger.error(f'Spitch ASR error: {e}')
            raise HTTPException(status_code=502, detail=f'ASR service error: {str(e)}')

    def synthesize(self, text: str, language: str = 'en', voice: str = 'sade') -> bytes:
        """Try Spitch first, fall back to gTTS if unavailable."""
        # Attempt 1: Spitch
        try:
            response = self.client.speech.generate(
                text=text,
                language=language,
                voice=voice
            )
            if isinstance(response, bytes):
                return response
            if hasattr(response, 'read'):
                return response.read()
            if hasattr(response, 'content'):
                return response.content
            return bytes(response)
        except Exception as e:
            logger.warning(f'Spitch TTS unavailable, falling back to gTTS: {e}')

        # Attempt 2: gTTS fallback
        try:
            from gtts import gTTS
            gtts_lang = language if language != 'pcm' else 'en'
            tts = gTTS(text=text, lang=gtts_lang)
            buffer = io.BytesIO()
            tts.write_to_fp(buffer)
            buffer.seek(0)
            return buffer.read()
        except Exception as e:
            logger.error(f'gTTS fallback also failed: {e}')
            raise HTTPException(status_code=502, detail=f'TTS service error: {str(e)}')


# VOICE MAP
VOICE_MAP = {
    'en': 'amina',
    'yo': 'sade',
    'ha': 'amina',
    'ig': 'amina',
    'pcm': 'amina',
    'fr': 'amina',
    'sw': 'amina',
    'am': 'amina',
    'ar': 'amina',
}

# create the service objects
translation_service = GroqTranslationService(GROQ_API_KEY)
audio_service = SpitchAudioService(SPITCH_API_KEY)

print('Service layer completed')

In [ ]:
app = FastAPI(
    title='Local Languages AI API',
    description=(
        'Production-ready API for African and local language processing: '
        'machine translation (Groq LLM), automatic speech recognition, '
        'and text-to-speech synthesis (Spitch).'
    ),
    version='1.0.0',
    docs_url='/docs',
    redoc_url='/redoc',
)

# Endpoint 1: Health Check
@app.get('/health', response_model=HealthResponse, tags=['System'])
async def health_check():
    """Check API and downstream service status."""
    return HealthResponse(
        status='healthy',
        version='1.0.0',
        services={
            'groq': 'configured' if GROQ_API_KEY else 'missing key',
            'spitch': 'configured' if SPITCH_API_KEY else 'missing key',
        },
    )

# Endpoint 2: Translation
@app.post('/translate', response_model=TranslationResponse, tags=['Translation'])
async def translate_text(request: TranslationRequest):
    if request.source_language == request.target_language:
        raise HTTPException(status_code=400, detail='Source and target languages must be different.')

    translated = translation_service.translate(
        text=request.text,
        source_lang=request.source_language.value,
        target_lang=request.target_language.value,
    )

    return TranslationResponse(
        source_language=request.source_language.value,
        target_language=request.target_language.value,
        original_text=request.text,
        translated_text=translated,
        model_used=translation_service.model,
    )

# Endpoint 3: Speech-to-Text
@app.post('/transcribe', response_model=TranscriptionResponse, tags=['Speech-to-Text'])
async def transcribe_audio(
    file: UploadFile = File(..., description='Audio file (WAV, MP3, OGG, FLAC)'),
    language: str = Form(default='en', description='Language code'),
):
    allowed_types = {
        'audio/wav', 'audio/mpeg', 'audio/ogg', 'audio/flac',
        'audio/x-wav', 'audio/mp3', 'audio/wave'
    }
    if file.content_type and file.content_type not in allowed_types:
        raise HTTPException(status_code=400, detail=f'Unsupported format: {file.content_type}')

    audio_bytes = await file.read()

    if len(audio_bytes) > 25 * 1024 * 1024:
        raise HTTPException(status_code=400, detail='File exceeds 25 MB limit.')

    # only pass (audio_bytes, language) -- not filename
    result = audio_service.transcribe(audio_bytes, language)

    return TranscriptionResponse(
        transcribed_text=result['transcribed_text'],
        language=result.get('language', language),
    )

# Endpoint 4: Text-to-Speech
@app.post('/synthesize', tags=['Text-to-Speech'])
async def synthesize_speech(request: SynthesisRequest):
    voice = VOICE_MAP.get(request.language, 'sade')
    audio_bytes = audio_service.synthesize(request.text, request.language, voice)

    audio_b64 = base64.b64encode(audio_bytes).decode('utf-8')

    return SynthesisResponse(
        audio_base64=audio_b64,
        content_type='audio/wav',
        message='Speech synthesized successfully',
    )

# Catch-all error handler
@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    logger.error(f'Unhandled error: {exc}')
    return JSONResponse(
        status_code=500,
        content={'detail': 'An internal server error occurred. Please try again.'},
    )


print('FastAPI app created with 4 endpoints')

In [ ]:
# list of language names for dropdowns
LANGUAGE_CHOICES = [lang.value for lang in SupportedLanguage]

# map full names to short codes that Spitch uses
LANG_CODE_MAP = {
    'English': 'en',
    'Yoruba': 'yo',
    'Hausa': 'ha',
    'Igbo': 'ig',
    'Nigerian Pidgin': 'pcm',
    'French': 'fr',
    'Swahili': 'sw',
    'Amharic': 'am',
    'Arabic': 'ar',
}

def gr_translate(text: str, source_lang: str, target_lang: str) -> str:
    if not text.strip():
        return 'Please enter some text to translate.'
    if source_lang == target_lang:
        return 'Source and target languages must be different.'
    try:
        result = translation_service.translate(text.strip(), source_lang, target_lang)
        return result
    except Exception as e:
        return f'Error: {str(e)}'


def gr_transcribe(audio_filepath: str, language: str) -> str:
    if audio_filepath is None:
        return 'Please upload or record an audio file.'
    try:
        lang_code = LANG_CODE_MAP.get(language, 'en')
        with open(audio_filepath, 'rb') as f:
            audio_bytes = f.read()
        result = audio_service.transcribe(audio_bytes, lang_code)
        return result['transcribed_text']
    except Exception as e:
        return f'Error: {str(e)}'

def gr_synthesize(text: str, language: str) -> Optional[str]:
    if not text.strip():
        return None
    try:
        lang_code = LANG_CODE_MAP.get(language, 'en')
        voice = VOICE_MAP.get(lang_code, 'sade')
        audio_bytes = audio_service.synthesize(text.strip(), lang_code, voice)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
        tmp.write(audio_bytes)
        tmp.close()
        return tmp.name
    except Exception as e:
        logger.error(f'TTS error: {e}')
        return None


# GRADIO UI

with gr.Blocks(title='Local Languages AI API') as demo:

    gr.Markdown('# Local Languages AI API\n**Machine Translation \u00b7 Speech Recognition \u00b7 Text-to-Speech**')

    # Tab 1: Translation
    with gr.Tab('Translate'):
        gr.Markdown('Translate text between African and world languages.')
        with gr.Row():
            src_lang = gr.Dropdown(LANGUAGE_CHOICES, value='English', label='From')
            tgt_lang = gr.Dropdown(LANGUAGE_CHOICES, value='Yoruba', label='To')
        input_text = gr.Textbox(label='Text to Translate', placeholder='Type here...', lines=4)
        translate_btn = gr.Button('Translate', variant='primary')
        output_text = gr.Textbox(label='Translation', lines=4, interactive=False)
        translate_btn.click(gr_translate, inputs=[input_text, src_lang, tgt_lang], outputs=output_text)

    # Tab 2: Speech to Text
    with gr.Tab('Transcribe'):
        gr.Markdown('Upload or record audio to get text.')
        asr_lang = gr.Dropdown(LANGUAGE_CHOICES, value='English', label='Audio Language')
        audio_input = gr.Audio(label='Audio', type='filepath', sources=['upload', 'microphone'])
        transcribe_btn = gr.Button('Transcribe', variant='primary')
        asr_output = gr.Textbox(label='Transcription', lines=4, interactive=False)
        transcribe_btn.click(gr_transcribe, inputs=[audio_input, asr_lang], outputs=asr_output)

    # Tab 3: Text to Speech
    with gr.Tab('Synthesize'):
        gr.Markdown('Convert text into spoken audio.')
        tts_lang = gr.Dropdown(LANGUAGE_CHOICES, value='English', label='Language')
        tts_input = gr.Textbox(label='Text to Speak', placeholder='Type something...', lines=3)
        synth_btn = gr.Button('Synthesize Speech', variant='primary')
        audio_output = gr.Audio(label='Generated Audio', type='filepath')
        synth_btn.click(gr_synthesize, inputs=[tts_input, tts_lang], outputs=audio_output)


print('Gradio interface built')

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)


def test_health():
    r = client.get('/health')
    assert r.status_code == 200
    data = r.json()
    assert data['status'] == 'healthy'
    print(f'Health check: {data}')

def test_same_language_rejected():
    payload = {'text': 'Hello', 'source_language': 'English', 'target_language': 'English'}
    r = client.post('/translate', json=payload)
    assert r.status_code == 400
    print('Same-language rejection works')

def test_empty_text_rejected():
    payload = {'text': '   ', 'source_language': 'English', 'target_language': 'Hausa'}
    r = client.post('/translate', json=payload)
    assert r.status_code == 422
    print('Empty text rejection works')

def test_live_translation():
    payload = {'text': 'Good morning, how are you?', 'source_language': 'English', 'target_language': 'Yoruba'}
    r = client.post('/translate', json=payload)
    assert r.status_code == 200
    data = r.json()
    assert len(data['translated_text']) > 0
    print(f"Translation: '{data['original_text']}' -> '{data['translated_text']}'")


print('Running tests...\n')
test_health()
test_same_language_rejected()
test_empty_text_rejected()

if GROQ_API_KEY:
    test_live_translation()
else:
    print('Skipping live test (no Groq key)')

print('All tests passed!')

In [ ]:
# share=True creates a public URL for testing
# keep this tab open to keep the link active

demo.launch(share=True, show_error=True)